# Rapid eight games (v3)

Estimate a Lichess **rapid** rating from eight games by one player (10% / 50% / 90% band). Goal: beat rapid v2 on the same held-out players. Validation and test players match v2; score test once.

**Data:** Lichess `lichess_db_standard_rated_2026-09.pgn`, Elo 800–2400. See `docs/data.md`.

**Colab setup:** mount Drive, then:

- `MyDrive/src/` — full `src` package (`schema.py` at the root of that folder)
- `MyDrive/DATA/rapid2026/` — `player_games_v3.parquet`, `player_sample_v3_meta.json`, `player_games_v3_meta.json`

**Success (ship card):** test macro MAE ≤ 120 and tail bias within ±80 Elo. Median interval width and per-bin coverage are recorded; they do not block `ship_ready`.

**Gates before shipping the rapid card:**

- **G2 (validation):** chosen row's validation macro MAE ≤ 142.6 (at least 10 Elo better than v2's 152.6). If it fails, stop — do not score test.
- **G3 (test, once):** test macro MAE < 152.2 with clocks and with clocks stripped. Passing G3 can promote v3 even if `ship_ready` is still false (lab estimate).

v2 test macro MAE was 152.2.


In [ ]:
from pathlib import Path
import sys

if not Path("/content").exists():
    raise RuntimeError(
        "This notebook trains in Google Colab. Open notebooks/train_player8_v3.ipynb there "
        "and run every cell from the top."
    )

from google.colab import drive
import subprocess

drive.mount("/content/drive")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow>=14", "pandas>=2.0", "numpy>=1.24",
    "scikit-learn>=1.3", "scipy>=1.11", "xgboost>=2.0",
    "python-chess>=1.10", "matplotlib", "joblib",
])
if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Drive mount did not expose /content/drive/MyDrive")
print("[setup] Drive mounted")
print("[setup] Expecting player_games_v3.parquet, player_sample_v3_meta.json, and player_games_v3_meta.json in MyDrive/DATA/rapid2026")
print("[setup] Expecting schema.py in MyDrive/src")


In [ ]:
from pathlib import Path
import sys


def find_repo():
    path = Path("/content/drive/MyDrive/src")
    if not (path / "schema.py").exists():
        raise RuntimeError(
            "Upload the src package to MyDrive/src so schema.py sits at /content/drive/MyDrive/src/schema.py."
        )
    parent = path.parent
    resolved = str(parent.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"[setup] repo -> {resolved}")
    return parent


def find_data():
    path = Path("/content/drive/MyDrive/DATA/rapid2026")
    required = (
        "player_games_v3.parquet",
        "player_sample_v3_meta.json",
        "player_games_v3_meta.json",
    )
    missing = [name for name in required if not (path / name).exists()]
    if missing:
        raise RuntimeError(
            "Upload " + ", ".join(missing) + " to MyDrive/DATA/rapid2026. "
            "moves_v3.parquet stays on the local machine."
        )
    print(f"[setup] data -> {path}")
    return path


REPO = find_repo()
DATA = find_data()


In [ ]:
import pandas as pd

from src.train_v3 import load_v3

splits = load_v3(DATA)
for name in ("train", "val", "test"):
    frame = splits[name]
    print(f"[load] {name} players={frame['player_key'].nunique():,} rows={len(frame):,}")
    print(frame.groupby("elo_bin").size().rename("rows").to_string())
print("[load] train rows by subset")
print(splits["train"].groupby("subset").size().rename("rows").to_string())
print(f"[load] train rows with clocks blanked={int((splits['train']['has_clock'] == 0).sum()):,}")


## Baseline and ablation

`fit_v3` prints the engine-linear validation macro MAE, fits the pre-declared search, then fits each feature set in order: v2, engine and time control, opponent, clocks, then the full set with spread. The ablation is a report. It does not choose the features. The full set is what gets scored.

Selection is the lowest validation macro MAE. Within-150 does not choose the row.


In [ ]:
import pandas as pd

from src.train_v3 import fit_v3

fitted = fit_v3(splits)
print(fitted["ablation"].round(3).to_string(index=False))
print(pd.DataFrame(fitted["rows"]).round(3).to_string(index=False))
if "q50" in fitted["cap"]:
    raise RuntimeError("Raise n_estimators in FROZEN_PARAMS_V3 and run from the top.")
val_mae = float(fitted["chosen"]["val_macro_mae"])
g2_pass = val_mae <= 142.6
print(f"[gate] G2 {'pass' if g2_pass else 'fail'}: val macro MAE {val_mae:.1f} vs v2 152.6")
if not g2_pass:
    raise RuntimeError("G2 failed. Do not score test. Next step is Maia features (v4).")


## Test, scored once

The per-bin scales were chosen on the winning row's validation predictions. Re-running this cell to compare another configuration uses test as a tuning set. Run the notebook from the top instead.

G3 passes when test macro MAE is below 152.2 with clocks and also with the clock block stripped. `ship_ready` is printed as the fit left it.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.train_v3 import score_test_once, test_points

tested = score_test_once(fitted, splits)
print(pd.DataFrame(tested["summary"]["bins"]).round(2).to_string(index=False))
print(
    f"[test] no-clock macro MAE={tested['no_clock']['macro_mae']:.1f}  "
    f"ship_ready={tested['decision']['ship_ready']}"
)
clock_mae = float(tested["summary"]["macro_mae"])
plain_mae = float(tested["no_clock"]["macro_mae"])
g3_pass = clock_mae < 152.2 and plain_mae < 152.2
print(
    f"[gate] G3 {'pass' if g3_pass else 'fail'}: "
    f"test macro MAE {clock_mae:.1f} with clocks, {plain_mae:.1f} without, vs v2 152.2"
)

y_test, mid = test_points(fitted, splits)
fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(y_test, mid, s=12, alpha=0.35, color="#2a6fbb")
lims = [float(min(y_test.min(), mid.min())), float(max(y_test.max(), mid.max()))]
ax.plot(lims, lims, color="black", lw=1)
ax.set_xlabel("Actual mean Elo")
ax.set_ylabel("Predicted median Elo")
ax.set_title("Test players: median vs actual Elo")
fig.tight_layout()
out_dir = Path("/content/drive/MyDrive/DATA/rapid8v3")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "pred_vs_actual.png", bbox_inches="tight")
plt.show()
print("[plot] saved pred_vs_actual.png. Points above the diagonal are guessed too high.")


In [ ]:
import matplotlib.pyplot as plt

from src.train_v3 import validation_report

importance, top = validation_report(fitted, splits)
print("[features] top of the validation permutation ranking")
print(importance.head(12).round(3).to_string(index=False))
print()
for rank, item in enumerate(top, start=1):
    print(f"{rank}. {item['feature']}  (validation MAE rises by {item['importance']:.2f} Elo when shuffled)")
    print(f"   {item['description']}")

show = importance.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.barh(show["feature"], show["importance"], color="#415a77")
ax.set_xlabel("Validation MAE increase (Elo)")
ax.set_ylabel("Feature")
ax.set_title("Permutation importance of the rapid v3 median")
fig.tight_layout()
fig.savefig(out_dir / "feature_importance.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.2))
flat = list(axes.ravel())
for axis, item in zip(flat, top):
    curve = item["curve"]
    if not curve:
        axis.set_axis_off()
        axis.set_title(item["feature"])
        continue
    xs = [point["value_mean"] for point in curve]
    ys = [point["elo_mean"] for point in curve]
    axis.plot(xs, ys, marker="o", color="#2a6fbb")
    axis.set_title(item["feature"])
    axis.set_xlabel("Feature value")
    axis.set_ylabel("Mean actual Elo")
flat[-1].set_axis_off()
fig.suptitle("Validation: mean actual Elo across each feature", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "feature_curves.png", bbox_inches="tight")
plt.show()
print("[plot] saved feature_importance.png and feature_curves.png")


In [ ]:
import json
from pathlib import Path

from src.train_v3 import save_v3

meta = json.loads((DATA / "player_games_v3_meta.json").read_text(encoding="utf-8"))
bundle = save_v3(Path("/content/drive/MyDrive/DATA/rapid8v3"), fitted, tested, splits, meta)
print(f"[save] {bundle}")


## Next steps

- If G2 failed, this notebook stopped before test. Do not look at test to pick a new row. The next model is Maia features (v4).
- If G3 failed, do not serve this bundle. Leave the rapid page on `rapid8-v2`. Maia is v4.
- If G3 passed, download `MyDrive/DATA/rapid8v3` to `model/rapid8-v3`. The page still shows one Elo. Swap the rapid card to `rapid8-v3` only after that download.
- Do not re-run the test cell to compare another configuration. Run the notebook from the top instead.
